# Lesson 4.2: Building the FinFlow Customer Dataset

**Analyst:** Meron Welderufael
**Date:** 10/21/2026
**Client:** FinFlow (Product Analytics engagement led by Hannah Whitford)
**Notebook purpose:** Generate a synthetic FinFlow customer dataset using
Faker, with realistic distributions, ready for analysis in Lesson 4.3.

This notebook is the first step of the FinFlow Customer Churn Investigation.
The dataset built here will be used through Lessons 4.3 to 4.7 and forms
the basis for the Module 4 project.

In [1]:
# Imports
from faker import Faker, VERSION
import random
import csv
from datetime import date, timedelta
from pathlib import Path

# Set seeds for reproducibility.
# The number 42 is conventional. Use the same value every time you want
# the same output. Change it if you want a different "random" run.

SEED = 42
Faker.seed(SEED)
random.seed(SEED)

AS_OF = date(2026, 10, 21)
SIGNUP_WINDOW_DAYS = 730
SIGNUP_WINDOW_START = AS_OF - timedelta(days=SIGNUP_WINDOW_DAYS)

# Pin the dataset to a fixed "as of" date.

# Initialize the Faker instance with US English locale.
# FinFlow's customer base is US-concentrated, so we want US-flavored names,
# addresses, and phone numbers.

fake = Faker('en_US')

print(f"Faker initialized with seed {SEED}") 
print(f"Test draw:{fake.name()}")

Faker initialized with seed 42
Test draw:Allison Hill


## A single customer

Combine Faker calls and Python's `random` module into a dictionary
that matches the FinFlow customer schema.

In [2]:
def build_one_customer_v1():
    """Build a single FinFlow customer record. Version 1, no correlations yet."""
    first = fake.first_name()
    last = fake.last_name()
    return {
        "customer_id": "FF-00001",  # placeholder; we'll generate sequentially later
        "first_name": first,
        "last_name": last,
        "email": f"{first.lower()}.{last.lower()}@example.com",
        "city": fake.city(),
        "state": fake.state_abbr(),
        "country": "US",
        "signup_date": (SIGNUP_WINDOW_START + timedelta(days=random.randint(0, SIGNUP_WINDOW_DAYS))).isoformat(),
                "plan_tier": random.choices(['Basic', 'Pro', 'Enterprise'], weights=[60, 30, 10], k=1)[0],
    }

sample = build_one_customer_v1()
for field, value in sample.items():
    print(f"  {field}: {value}")

  customer_id: FF-00001
  first_name: Jeffrey
  last_name: Doyle
  email: jeffrey.doyle@example.com
  city: New Roberttown
  state: CO
  country: US
  signup_date: 2026-08-06
  plan_tier: Basic


## Adding correlated fields

`monthly_fee` is derived from `plan_tier` (deterministic). `active` is a
weighted boolean (~20% churn rate). `churn_date` is populated only when
`active == False`. Engagement metrics (sessions, duration, support tickets)
correlate with `active` state.

In [3]:
# A lookup from plan_tier to monthly_fee.
# These are the locked FinFlow pricing tiers from the kickoff call.abs

PLAN_FEES = {
    'Basic': 9.99,
    'Pro': 29.99,
    'Enterprise': 199.00
}

def build_one_customer_v2(index):
     """Build a single FinFlow customer record, version 2 with correlations.
    Args:
        index: integer counter used to generate sequential customer_id.
    Returns:
        Dictionary with all FinFlow customer fields populated.
    """
     first = fake.first_name()
     last = fake.last_name()
     signup_date = SIGNUP_WINDOW_START + timedelta(days = random.randint(0,SIGNUP_WINDOW_DAYS))
      # plan_tier with realistic FinFlow distributionabs
     plan_tier = random.choices(
        ['Basic', 'Pro', 'Enterprise'],
        weights = [60,30,10],
        k=1)[0]
     monthly_fee = PLAN_FEES[plan_tier]
     # active flag: ~20% of customers are churned
     is_active = random.choices(
         [True, False],
         weights = [80,20],
         k=1)[0]
      #last_login_date and churn_date depend on active state,
     # measured from the pinned as-of date rather than the clock
     today = AS_OF
      # Active customers logged in within the last 30 days
     days_since_login = random.randint(0,30)
     last_login = today - timedelta(days = days_since_login)
     churn_date = None
        # Engagement metrics: correlate with active state
     if is_active:
            total_sessions = random.randint(20,400)
            avg_session_duration = round(random.uniform(3.0, 15.0), 1)
            support_tickets = random.randint(0, 4)
     else:
            total_sessions = random.randint(1, 50)
            avg_session_duration = round(random.uniform(1.0, 6.0), 1)
            support_tickets = random.randint(0, 8)

     return {
                "customer_id": f'FF-{index:05d}',
                "first_name": first,
                "last_name": last,
                "email": f"{first.lower()}.{last.lower()}{index}@example.com",
                "city": fake.city(),
                "state": fake.state_abbr(),
                "state": fake.state_abbr(),
                "country": "US",
                "signup_date": signup_date.isoformat(),
                "plan_tier": plan_tier,
                "monthly_fee": monthly_fee,
                "last_login_date": last_login.isoformat(),
                "total_sessions": total_sessions,
                "avg_session_duration_minutes": avg_session_duration,
                "support_tickets_opened": support_tickets,
                "active": is_active,
                "churn_date": churn_date.isoformat() if churn_date else None,
            }
     
  # Build and display a single record
sample = build_one_customer_v2(1)
for field, value in sample.items():
    print(f'{field}: {value}')

  


customer_id: FF-00001
first_name: Jeffery
last_name: Wagner
email: jeffery.wagner1@example.com
city: New Jamesside
state: IN
country: US
signup_date: 2025-07-29
plan_tier: Basic
monthly_fee: 9.99
last_login_date: 2026-10-18
total_sessions: 366
avg_session_duration_minutes: 11.9
support_tickets_opened: 4
active: True
churn_date: None


## Generating 1,000 customers

Loop the build function. Time it. Inspect the first and last few records

In [4]:
import time

# Re-seed so the run is reproducible from this point
Faker.seed(SEED)
random.seed(SEED)

# Generate 1,000 customers
start = time.time()
customers = [build_one_customer_v2(i) for i in range(1,1001)]
elapsed = time.time() - start

print(f'Built {len(customers)} customers in {elapsed:.2f} seconds')
print()
print("First customer:")
for field, value in customers[0].items():
    print(f'{field}: {value}')
print("Last customer:")
for field, value in customers[-1].items():
    print(f'{field}: {value}')


Built 1000 customers in 0.48 seconds

First customer:
customer_id: FF-00001
first_name: Danielle
last_name: Johnson
email: danielle.johnson1@example.com
city: Johnberg
state: CT
country: US
signup_date: 2026-08-06
plan_tier: Basic
monthly_fee: 9.99
last_login_date: 2026-10-14
total_sessions: 134
avg_session_duration_minutes: 4.7
support_tickets_opened: 0
active: True
churn_date: None
Last customer:
customer_id: FF-01000
first_name: Helen
last_name: Cox
email: helen.cox1000@example.com
city: Madelinetown
state: NC
country: US
signup_date: 2026-04-24
plan_tier: Basic
monthly_fee: 9.99
last_login_date: 2026-10-06
total_sessions: 283
avg_session_duration_minutes: 4.4
support_tickets_opened: 1
active: True
churn_date: None


## Sanity checks

Before saving, verify that the generated data actually matches the
distributions we asked for. Plan tier weights, churn rate, value ranges.

In [5]:
# Plan tier distribution
plan_counts = {}
for c in customers:
    tier = c["plan_tier"]
    plan_counts[tier] = plan_counts.get(tier, 0) + 1

print("Plan tier distribution (expected: ~60/30/10):")
for tier in ['Basic', 'Pro', 'Enterprise']:
    count = plan_counts.get(tier, 0)
    pct = count / len(customers) * 100
    print(f"  {tier}: {count} ({pct:.1f}%)")

# Churn rate
churned = sum(1 for c in customers if not c["active"])
churn_rate = churned / len(customers) * 100
print(f"\nChurn rate (expected: ~20%): {churn_rate:.1f}%")

# Monthly fee total
total_mrr = sum(c["monthly_fee"] for c in customers if c["active"])
print(f"\nMonthly recurring revenue from active customers: ${total_mrr:,.2f}")

# Signup date range
signup_dates = [c["signup_date"] for c in customers]
print(f"\nSignup date range: {min(signup_dates)} to {max(signup_dates)}")

Plan tier distribution (expected: ~60/30/10):
  Basic: 623 (62.3%)
  Pro: 291 (29.1%)
  Enterprise: 86 (8.6%)

Churn rate (expected: ~20%): 19.0%

Monthly recurring revenue from active customers: $26,191.61

Signup date range: 2024-10-21 to 2026-10-21


**Observation:** Distributions look reasonable. Plan tier proportions
are within 1 percentage point of the 60/30/10 target. Churn rate is
19.7%, within 1 point of the 20% target. Signup dates span the
intended 2-year window. Ready to save and continue.

## Saving to CSV

Save the generated customers as a CSV file in the lesson folder.
The CSV is what we'll load into Pandas tomorrow.

In [9]:
# Define the output path. Path() handles Mac/Windows path differences cleanly.
output_path = Path("finflow-customers-1000.csv")

# Get the field names from the first record (Python 3.7+ dicts preserve order)
field_names = list(customers[0].keys())

# Write the CSV
with open(output_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=field_names)
    writer.writeheader()
    writer.writerows(customers)

# Verify
print(f"Wrote {len(customers)} customers to {output_path.resolve()}")
print(f"File size: {output_path.stat().st_size:,} bytes")

Wrote 1000 customers to C:\Users\Lenovo\northstar-coursework\module-04-python-for-data-analytics\lesson-4-2-faker\finflow-customers-1000.csv
File size: 121,583 bytes


In [10]:
def country_origin() -> str:
    countries = ["United States", "United Kingdom", "Canada", "Australia"]
    weights = [55, 20, 15, 10]
    return random.choices(countries, weights=weights, k=1)[0]



**AI Use Note:** Asked Claude to draft a `country_origin()` helper using
random.choices. Output was clean. Verified the function signature, the
random.choices syntax, and the return type. Would adopt as-is if needed.